In [0]:
%run ./01-ATSConfigs

In [0]:
%run ./03-Endpoints

In [0]:
%run ./04-VectorSearch

In [0]:
class ProfileInjestion:
    def __init__(self):
        pass

    def get_source(self):
        source_dir = (
            spark.sql(
                f"""select date_add(last_load_date,1) as source_dir
                from {conf.jobs_metadata_table_name}
                where job_name = '{conf.profile_ingestion_job_name}'
                order by last_load_date desc"""
            ).first(1)
            .asDict()['source_dir']
            .strftime("%Y-%m-%d")
            )
        return source_dir
    
    def cleanup_destination_dir(self,dest_path):
        print("cleaningup {dest_path}..", end="")
        dbutils.fs.rm(dest_path,recurse=True)
        dbutils.fs.mkdirs(dest_path)
        print("done")

    def update_metadata(self,ingestion_date):
        print("updating metadata..", end="")
        spark.sql(
            f"""insert into {conf.jobs_metadata_table_name}
            values('{conf.profile_ingestion_job_name}','{ingestion_date}',
            current_timestamp(),
            "Job Execution" )"""
        )

    def ingest_profiles(self):
        import requests
        from concurrent.futures import ThreadPoolExecutor
        import collections

        download_dir = self.get_source()

        dest_path = f"/Volumes/{conf.catalog}/{conf.db}/{conf.profile_landing_zone}/{download_dir}"
        self.cleanup_destination_dir(dest_path)
        api_url = f"https://api.github.com/repos/{conf.owner}/{conf.repo}/contents/{conf.profile_source}/{download_dir}"
        files = requests.get(api_url).json()
        download_urls = [file["download_url"] for file in files]

        def download_files(download_url):
            filename = download_url.split("/")[-1]
            with requests.get(download_url, stream=True) as r:
                with open(f"{dest_path}/{filename}", "wb") as f:
                    for chunk in r.iter_content(chunk_size=8 * 1024):
                        f.write(chunk)

        print(f"Downloading profiles...", end="")
        with ThreadPoolExecutor(max_workers=4) as executors:
            collections.deque(executors.map(download_files, download_urls))
        self.update_metadata(download_dir)
        print("Done")
        return dest_path
    
    def assert_file_count(self, dir_name, expected_count):
        print(f"Validating ingested file counts in {dir_name}...", end='')
        files = dbutils.fs.ls(f"/Volumes/{conf.catalog}/{conf.db}/{conf.profile_landing_zone}/{dir_name}")
        actual_count = len([file.path for file in files])
        assert actual_count == expected_count, f"Expected {expected_count:,} files, found {actual_count:,} in {dir_name}" 
        print(f"Found {actual_count:,} / Expected {expected_count:,} files: Success")

    def validate(self, iter):
        import time
        start = int(time.time())
        print(f"\nValidating profile ingestion into landing zone...")
        self.assert_file_count("2025-07-01" if iter == 1 else "2025-07-02", 5)
        print(f"Validating profile load into bronze layer completed in {int(time.time()) - start} seconds")
